# qec-geometry · interactive demo

**Geometric diagnostics for quantum error-correcting codes**

- **A0/A1 topology** of decoding-error patterns — A1 − A0 = code distance
- **Closed-form thresholds** — p_L ≈ A·p², p_th = 1/A = 6/[n(n−1)]
- **Anyon typing** — Ising / Majorana statistics

| | |
|---|---|
| GitHub | https://github.com/sdoygb/qec-geometry |
| PyPI | `pip install qec-geometry` |
| DOI | https://doi.org/10.5281/zenodo.21946024 |

Run every cell in order (**Run All**). The surface-code cell (last) takes ~20 s.

In [ ]:
from qecgeo.codes import ALL, steane_code
from qecgeo.threshold import analyze_eta, verify_quadratic, concatenation_sequence
from qecgeo.error_geometry import run_error_geometry
import qecgeo

print("qecgeo", qecgeo.__version__, "— imports OK")

## 1 · Code construction

Four geometric stabilizer codes from the framework (10.27): the [[5,1,3]] five-qubit code, Steane [[7,1,3]], Shor [[9,1,3]], and RM(1,4) [[15,7,3]].

In [ ]:
for make in ALL:
    c = make()
    print(f"{c.name:38s} n={c.n}  k={c.k}  stabilizers={c.m}")

## 2 · Closed-form threshold (exact enumeration)

For a code with n qubits, the weight-2 error analysis gives the logical error rate
p_L ≈ A·p² with A = η·C(n,2), where η is the exact fraction of weight-2 errors that
misrecover to a logical operator. The ideal concatenation threshold is p_th = 1/A.

In [ ]:
r = analyze_eta(steane_code())
print(f"n = {r['n']}   weight-2 errors = {r['total']}")
print(f"η (misrecovery fraction) = {r['eta']:.6f}")
print(f"A = η·C(n,2)            = {r['A']:.6f}")
print(f"p_th = 1/A              = {r['p_th']:.4%}")
print(f"  (6/[n(n−1)] = {6/(7*6):.4%} — closed form)")

## 3 · Threshold table (all codes)

In [ ]:
print(f"{'code':38s} {'n':>3s} {'η':>8s} {'A':>8s} {'p_th':>10s}")
for make in ALL:
    c = make()
    r = analyze_eta(c)
    print(f"{c.name:38s} {r['n']:3d} {r['eta']:8.4f} {r['A']:8.2f} {r['p_th']:10.4%}")

## 4 · Quadratic law p_L ≈ A·p² — Monte Carlo check

Each qubit suffers independent X/Y/Z noise (p/3 each); single-round lookup-table
recovery, 300 000 shots per p.

In [ ]:
for row in verify_quadratic(steane_code()):
    print(f"p={row['p']:.2f}   pL={row['pL']:.5f}   A·p²={row['Ap2']:.5f}   ratio={row['ratio']:.2f}")

## 5 · Concatenation compression below threshold

Starting from p0 = 1% (below p_th ≈ 14.3%), each concatenation level squares the error rate.

In [ ]:
A = analyze_eta(steane_code())['A']
seq = concatenation_sequence(A, p0=0.01, levels=6)
for i, p in enumerate(seq):
    print(f"level {i}:  p_L = {p:.3e}")
print("→ error rate collapses to 0 after 4 concatenations")

## 6 · A0/A1 topology of error patterns (surface code L=4)

Decoding errors split into **A0** (contractible, correctable) and **A1**
(non-contractible — these ARE the logical errors). The crossing criterion
(error chain touching both opposite boundaries) is the topological signature
of A1. Feature ratios A1/A0 from 5 000 shot syndromes via stim + PyMatching.

In [ ]:
res = run_error_geometry(L=4, rounds=3, noise=0.03, shots=5000, with_edges=True)
print(f"logical error rate p_L = {res['pL']:.4f}")
print(f"crossing-rate lift A1/A0 = {res['cross_lift']:.2f}×")
print()
print("feature ratios (A1 vs A0):")
for r in res['ratios']:
    print(f"  {r['field']:16s}  ratio = {r['ratio']:7.2f}  {r['verdict']}")

---
*Geometric theory framework: github.com/sdoygb/conjugate-spectral-geometry · MIT license*